# Bloco 3: O LLM como instrumento de medição
**Anotação com modelo aberto local (Ollama) ou comercial, sempre com cache e versionamento.**

Em sala **não** precisamos rodar a anotação completa: as anotações já estão em `dados/cache_llm/` (geradas com `python/anotar_llm.py`). Este notebook mostra o mecanismo e permite testar outro modelo ou prompt.

Cada chamada fica registrada com modelo, prompt, hash do prompt, temperatura, semente, resposta bruta e tempo. Se o processo for interrompido, ele continua de onde parou.

In [1]:
# === Configuração: funciona localmente e no Google Colab ===
import os, sys, subprocess
from pathlib import Path

# PROVISÓRIO: endereço do repositório público do minicurso (ajustar quando publicado)
REPO_URL = "https://github.com/jodavid/minicurso-ia-medicao"

EM_COLAB = "google.colab" in sys.modules
if EM_COLAB:
    if not Path("minicurso-ia-medicao").exists():
        subprocess.run(["git", "clone", "-q", REPO_URL], check=True)
    RAIZ = Path("minicurso-ia-medicao").resolve()
else:
    # localmente: a raiz é a pasta do minicurso (este notebook fica em python/)
    RAIZ = Path.cwd().parent if Path.cwd().name == "python" else Path.cwd()

os.chdir(RAIZ)
sys.path.insert(0, str(RAIZ / "python"))
SAIDA = RAIZ / "resultados" / "python"          # o R grava em resultados/; o Python, aqui
(SAIDA / "figuras").mkdir(parents=True, exist_ok=True)
(SAIDA / "tabelas").mkdir(parents=True, exist_ok=True)
print("Raiz do projeto:", RAIZ)

Raiz do projeto: ~/minicurso_IA_medicao_publico


## 1. (Só no Colab) Instalar e iniciar o Ollama

Use um ambiente com **GPU** (Ambiente de execução → Alterar o tipo → T4 GPU). O instrumento é o **S5-nano** (Maritaca AI) em modo *instruct*, quantizado em 4 bits: o download tem ~3,5 GB e leva alguns minutos. O `ollama create` dá a ele o nome local `s5-nano-instruct` (ver `modelos/Modelfile.s5-nano-instruct`). (Por que este modelo? É um modelo brasileiro, derivado do Qwen3.5-4B e treinado com dados em português; veja o capítulo 5 do livro.)

In [2]:
if EM_COLAB:
    subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)
    subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    import time; time.sleep(5)
    subprocess.run(["ollama", "pull", "hf.co/maritaca-ai/S5-nano-GGUF:Q4_K_M"], check=True)
    subprocess.run(["ollama", "create", "s5-nano-instruct", "-f", "modelos/Modelfile.s5-nano-instruct"], check=True)

## 2. O Ollama está respondendo?

In [3]:
import requests
OLLAMA = os.environ.get("OLLAMA_HOST", "http://localhost:11434")
try:
    modelos = [m["name"] for m in requests.get(f"{OLLAMA}/api/tags", timeout=5).json()["models"]]
    print("Modelos disponíveis:", modelos)
except Exception as e:
    print("Ollama indisponível:", e)

Modelos disponíveis: ['s5-nano-instruct:latest']


## 3. Os prompts são parte do instrumento

Mudar uma vírgula pode mudar a medida. Por isso os prompts estão versionados em `prompts/` e cada anotação guarda o hash SHA-256 do texto.

In [4]:
import anotar_llm as al
for pid, arq in al.PROMPTS.items():
    texto = arq.read_text(encoding="utf-8")
    print(f"{pid}: {arq.name}  sha={al.sha(texto)}  ({len(texto)} caracteres)")
print()
print(al.PROMPTS["v2"].read_text(encoding="utf-8"))

v1: prompt_v1_simples.txt  sha=5eb0f92ea13d  (252 caracteres)
v2: prompt_v2_codebook.txt  sha=c4b5a5ade801  (1522 caracteres)
v3: prompt_v3_fewshot.txt  sha=6fdbfaf1d613  (2485 caracteres)

Você é um codificador de pesquisa em Ciência Política. Classifique a ementa de um projeto de lei brasileiro segundo o livro de códigos abaixo. Use APENAS o texto da ementa.

CATEGORIAS (escolha exatamente uma):
- PUNIR: o instrumento central é penal: cria crime, tipifica conduta, criminaliza, aumenta ou qualifica pena, altera o Código Penal, a parte penal do ECA ou cria crime eleitoral.
- REGULAR: estabelece regras não penais para o uso de IA: princípios, deveres, obrigações, proibições administrativas, direitos, transparência, rotulagem, auditoria, responsabilidade civil ou administrativa, proteção de dados, direitos autorais, proteção do consumidor ou do trabalhador, campanhas de prevenção.
- PROMOVER: fomenta ou adota IA: incentivos, fundos, pesquisa, ensino de IA em currículos, capacitação, ou u

## 4. Anotar algumas ementas

Usamos uma **semente diferente** da usada no material (42), para não misturar o seu experimento com o cache oficial. Troque `MODELO`, `PROMPT`, `TEMPERATURA` e observe como os rótulos mudam.

In [5]:
import pandas as pd
from medicao import carrega_corpus

MODELO, PROMPT, TEMPERATURA, SEMENTE, N_TESTE = "s5-nano-instruct", "v2", 0.0, 7, 5

corpus = carrega_corpus(RAIZ)
arq = al.anota(corpus.head(N_TESTE), MODELO, PROMPT, TEMPERATURA, SEMENTE)
res = pd.read_json(arq, lines=True)
res.merge(corpus[["doc_id", "ementa"]], on="doc_id")[["doc_id", "categoria_llm", "eleitoral_llm", "segundos", "ementa"]]

[s5-nano-instruct__v2__t0.0__s7] 0 em cache, 5 pendentes


,doc_id,categoria_llm,eleitoral_llm,segundos,ementa
0,doc0001,REGULAR,False,0.347,Estabelece os princípios para o uso da Intelig...
1,doc0002,REGULAR,False,0.338,Institui a Política Nacional de Inteligência A...
2,doc0003,REGULAR,False,0.335,"Cria a Lei da Inteligência Artificial, e dá ou..."
3,doc0004,REGULAR,False,0.338,Dispõe sobre o uso da Inteligência Artificial.
4,doc0005,REGULAR,False,0.338,"Dispõe sobre os princípios, direitos e obrigaç..."


Note a coluna `resposta_bruta`: modelos pequenos às vezes devolvem o JSON entre crases (```` ```json ````) ou com texto extra. Respostas fora do formato viram `None`, e a **taxa de falhas de formato** também é uma propriedade do instrumento.

In [6]:
res[["doc_id", "resposta_bruta"]].head()

,doc_id,resposta_bruta
0,doc0001,"{""categoria"": ""REGULAR"", ""eleitoral"": false}"
1,doc0002,"{""categoria"": ""REGULAR"", ""eleitoral"": false}"
2,doc0003,"{""categoria"": ""REGULAR"", ""eleitoral"": false}"
3,doc0004,"{""categoria"": ""REGULAR"", ""eleitoral"": false}"
4,doc0005,"{""categoria"": ""REGULAR"", ""eleitoral"": false}"


## 5. O plano de experimentos do material

O cache oficial foi gerado com `python python/anotar_llm.py --todas`:

| Execução | Prompt | Temperatura | Semente | Para quê |
|---|---|---|---|---|
| 1 | v1 (só nomes das categorias) | 0 | 42 | linha de base do LLM |
| 2 | v2 (livro de códigos) | 0 | 42 | validade |
| 3 | v3 (livro de códigos + exemplos) | 0 | 42 | efeito do *few-shot* |
| 4–6 | v2 | 0,8 | 1, 2, 3 | teste-reteste (confiabilidade) |

In [7]:
from medicao import carrega_llm
llm = carrega_llm(RAIZ)
llm.groupby("run_id").agg(n=("doc_id", "size"), falhas=("categoria_llm", lambda s: s.isna().mean()),
                          seg_mediano=("segundos", "median"))

,n,falhas,seg_mediano
run_id,,,
s5-nano-instruct__v1__t0.0__s42,304,0.0,0.359
s5-nano-instruct__v2__t0.0__s42,304,0.0,0.407
s5-nano-instruct__v2__t0.0__s7,5,0.0,0.338
s5-nano-instruct__v2__t0.8__s1,304,0.0,0.356
s5-nano-instruct__v2__t0.8__s2,304,0.0,0.358
s5-nano-instruct__v2__t0.8__s3,304,0.0,0.359
s5-nano-instruct__v3__t0.0__s42,304,0.0,0.439


## 6. (Opcional) API comercial

A chave **nunca** vai no código nem no GitHub: use variável de ambiente (no Colab, em *Secrets*). APIs comerciais trocam a versão do modelo sem aviso: registre a data, o identificador exato do modelo e guarde as respostas brutas.

In [8]:
if os.environ.get("OPENAI_API_KEY"):
    from openai import OpenAI
    cliente = OpenAI()
    prompt = al.PROMPTS["v2"].read_text(encoding="utf-8").replace("{{ementa}}", corpus["ementa"][0])
    r = cliente.chat.completions.create(
        model="gpt-4.1-mini", temperature=0, seed=42,
        response_format={"type": "json_object"},
        messages=[{"role": "user", "content": prompt}])
    print(r.model, r.choices[0].message.content)
else:
    print("OPENAI_API_KEY não definida: célula ignorada.")

OPENAI_API_KEY não definida: célula ignorada.
